# CSV and Excel Data Processing

CSV and Excel files are common sources of structured data.

Unlike plain text or PDFs, this type of data already has a clear structure with rows and columns.

In this notebook, I’m learning how to load CSV and Excel files and convert their data into a format that can be used in a RAG pipeline.

I’m also comparing simple loaders with custom processing so I can understand how much control I have over the final documents.

The main flow I’m working with is:

**CSV/Excel → Read Structured Data → Create Documents → Add Metadata → Prepare for RAG**

In [1]:
import pandas as pd
import os

I’m using Pandas here because it gives me a simple way to work with structured data.

I’ll use it to create sample files first and later to process the Excel data.

## Creating a Folder for Structured Data

I’m keeping the CSV and Excel files in a separate folder.

This makes the project easier to organize because I’ll be working with different types of data in different notebooks.

In [2]:
os.makedirs("E:/RAG/data/course_samples/structured_files", exist_ok=True)

## Creating Sample Data

Before loading real files, I’m creating a small dataset that I can use for testing.

The data contains product information such as:

- Product name
- Category
- Price
- Stock
- Description

I’m using the same data to create both a CSV file and an Excel file.

In [3]:
# Create sample data
data = {
    'Product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Webcam'],
    'Category': ['Electronics', 'Accessories', 'Accessories', 'Electronics', 'Electronics'],
    'Price': [999.99, 29.99, 79.99, 299.99, 89.99],
    'Stock': [50, 200, 150, 75, 100],
    'Description': [
        'High-performance laptop with 16GB RAM and 512GB SSD',
        'Wireless optical mouse with ergonomic design',
        'Mechanical keyboard with RGB backlighting',
        '27-inch 4K monitor with HDR support',
        '1080p webcam with noise cancellation'
    ]
}

# Save as CSV
df = pd.DataFrame(data)

df.to_csv(
    'E:/RAG/data/course_samples/structured_files/products.csv',
    index=False
)

## Creating an Excel File

Now I’m saving the same data into an Excel workbook.

This time I’m also creating a second sheet called `Summary`.

This is useful because Excel files can contain multiple sheets, and I need to understand how that structure can be preserved during ingestion.

In [4]:
# Save as Excel with multiple sheets
with pd.ExcelWriter(
    'E:/RAG/data/course_samples/structured_files/inventory.xlsx'
) as writer:

    df.to_excel(
        writer,
        sheet_name='Products',
        index=False
    )
    # Add another sheet
    summary_data = {
        'Category': ['Electronics', 'Accessories'],
        'Total_Items': [3, 2],
        'Total_Value': [1389.97, 109.98]
    }

    pd.DataFrame(summary_data).to_excel(
        writer,
        sheet_name='Summary',
        index=False
    )

## CSV Processing

Now I’m moving to the actual ingestion part.

CSV files are basically structured tables, so I want to see how a CSV loader converts each row into a document.

After that, I’ll create my own processing function to have more control over the document content and metadata.

In [5]:
from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import UnstructuredCSVLoader

C:\Users\sniti\AppData\Local\Temp\ipykernel_5288\731168459.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


## 1. Using CSVLoader

First, I’m using LangChain's `CSVLoader`.

The important thing I want to check is how the loader represents each row.

I’ll look at:

- Number of documents created
- Content of the first document
- Metadata attached to it

In [6]:
# Method 1: CSVLoader - Each row becomes a document
print("1️⃣ CSVLoader - Row-based Documents")

csv_loader = CSVLoader(
    file_path='E:/RAG/data/course_samples/structured_files/products.csv',
    encoding='utf-8',
    csv_args={
        'delimiter': ',',
        'quotechar': '"',
    }
)

csv_docs = csv_loader.load()

print(csv_docs)
print(f"Loaded {len(csv_docs)} documents (one per row)")

print("\nFirst document:")
print(f"Content: {csv_docs[0].page_content}")
print(f"Metadata: {csv_docs[0].metadata}")

1️⃣ CSVLoader - Row-based Documents
[Document(metadata={'source': 'E:/RAG/data/course_samples/structured_files/products.csv', 'row': 0}, page_content='Product: Laptop\nCategory: Electronics\nPrice: 999.99\nStock: 50\nDescription: High-performance laptop with 16GB RAM and 512GB SSD'), Document(metadata={'source': 'E:/RAG/data/course_samples/structured_files/products.csv', 'row': 1}, page_content='Product: Mouse\nCategory: Accessories\nPrice: 29.99\nStock: 200\nDescription: Wireless optical mouse with ergonomic design'), Document(metadata={'source': 'E:/RAG/data/course_samples/structured_files/products.csv', 'row': 2}, page_content='Product: Keyboard\nCategory: Accessories\nPrice: 79.99\nStock: 150\nDescription: Mechanical keyboard with RGB backlighting'), Document(metadata={'source': 'E:/RAG/data/course_samples/structured_files/products.csv', 'row': 3}, page_content='Product: Monitor\nCategory: Electronics\nPrice: 299.99\nStock: 75\nDescription: 27-inch 4K monitor with HDR support'), Do

### What I noticed

`CSVLoader` creates one LangChain `Document` for each row.

This is useful when each row represents an independent record.

For example, if I want to search for a particular product, a row-based document can work well.

The source and row information can also be kept in metadata.

## 2. Custom CSV Processing

Now I want more control over how each row is converted into a document.

Instead of depending completely on the default loader format, I can decide:

- What the document text should look like
- Which fields should be included
- What metadata should be stored
- How the information should be organized

This can be useful when the default loader output is not exactly what my RAG pipeline needs.

In [7]:
from typing import List
from langchain_core.documents import Document

In [8]:
# Method 2: Custom CSV processing for better control
print("\n2️⃣ Custom CSV Processing")


def process_csv_intelligently(filepath: str) -> List[Document]:
    """Process CSV with intelligent document creation"""
    df = pd.read_csv(filepath)
    documents = []

    # Strategy 1: One document per row with structured content
    for idx, row in df.iterrows():
        # Create structured content
        content = f"""Product Information:
        Name: {row['Product']}
        Category: {row['Category']}
        Price: ${row['Price']}
        Stock: {row['Stock']} units
        Description: {row['Description']}"""

        # Create document with rich metadata
        doc = Document(
            page_content=content,
            metadata={
                'source': filepath,
                'row_index': idx,
                'product_name': row['Product'],
                'category': row['Category'],
                'data_type': 'csv_row'
            }
        )

        documents.append(doc)

    return documents


2️⃣ Custom CSV Processing


In [9]:
process_csv_intelligently(
    'E:/RAG/data/course_samples/structured_files/products.csv')

[Document(metadata={'source': 'E:/RAG/data/course_samples/structured_files/products.csv', 'row_index': 0, 'product_name': 'Laptop', 'category': 'Electronics', 'data_type': 'csv_row'}, page_content='Product Information:\n        Name: Laptop\n        Category: Electronics\n        Price: $999.99\n        Stock: 50 units\n        Description: High-performance laptop with 16GB RAM and 512GB SSD'),
 Document(metadata={'source': 'E:/RAG/data/course_samples/structured_files/products.csv', 'row_index': 1, 'product_name': 'Mouse', 'category': 'Accessories', 'data_type': 'csv_row'}, page_content='Product Information:\n        Name: Mouse\n        Category: Accessories\n        Price: $29.99\n        Stock: 200 units\n        Description: Wireless optical mouse with ergonomic design'),
 Document(metadata={'source': 'E:/RAG/data/course_samples/structured_files/products.csv', 'row_index': 2, 'product_name': 'Keyboard', 'category': 'Accessories', 'data_type': 'csv_row'}, page_content='Product Infor

## Testing the Custom CSV Processor

Now I’m running my custom processing function on the CSV file.

I want to check the documents created by my own processing logic and compare them with the default `CSVLoader` output.

In [10]:
csv_custom_docs = process_csv_intelligently(
    'E:/RAG/data/course_samples/structured_files/products.csv'
)

print(f"Created {len(csv_custom_docs)} documents")

print("\nFirst document:")
print(csv_custom_docs[0].page_content)
print("\nMetadata:")
print(csv_custom_docs[0].metadata)

Created 5 documents

First document:
Product Information:
        Name: Laptop
        Category: Electronics
        Price: $999.99
        Stock: 50 units
        Description: High-performance laptop with 16GB RAM and 512GB SSD

Metadata:
{'source': 'E:/RAG/data/course_samples/structured_files/products.csv', 'row_index': 0, 'product_name': 'Laptop', 'category': 'Electronics', 'data_type': 'csv_row'}


## Comparing CSV Processing Strategies

There are two different approaches I tried here.

The first one is simple and works directly with the CSV structure.

The second one gives me more control over how the document is created.

The important thing I’m learning is that structured data does not always need to be converted into documents in exactly the same way.

The best approach depends on what I want to retrieve later.

In [11]:
# 📊 CSV Processing Strategies
print("\n📊 CSV Processing Strategies:")
print("\n1. Row-based (CSVLoader):")
print("  ✅ Simple one-row-one-document")
print("  ✅ Good for record lookups")
print("  ❌ Can Lose some table context")

print("\n2. Intelligent Processing:")
print("  ✅ Can Preserves relationships between fields")
print("  ✅ More control over document content")
print("  ✅ Creates summaries")
print("  ✅ Rich metadata")
print("  ✅ Better control for Q&A use cases")


📊 CSV Processing Strategies:

1. Row-based (CSVLoader):
  ✅ Simple one-row-one-document
  ✅ Good for record lookups
  ❌ Can Lose some table context

2. Intelligent Processing:
  ✅ Can Preserves relationships between fields
  ✅ More control over document content
  ✅ Creates summaries
  ✅ Rich metadata
  ✅ Better control for Q&A use cases


## Comparing CSVLoader with UnstructuredCSVLoader

I already loaded the CSV using `CSVLoader`.

Now I want to try `UnstructuredCSVLoader` on the same file and compare the output.

The main thing I want to understand is how the same CSV can be represented differently depending on the loader.

With `CSVLoader`, I get row-based documents.

With `UnstructuredCSVLoader`, I can preserve the CSV more like a table structure.

This comparison will help me understand which representation may be more useful for different RAG use cases.

## Loading the CSV as a Table

I’m using `mode="elements"` here because I want to see how Unstructured represents the CSV table.

I’m using the same `products.csv` file so the comparison with `CSVLoader` is fair.

In [12]:
print("\n3️⃣ UnstructuredCSVLoader - Table Based Processing")

unstructured_csv_loader = UnstructuredCSVLoader(
    "E:/RAG/data/course_samples/structured_files/products.csv",
    mode="elements"
)

unstructured_csv_docs = unstructured_csv_loader.load()

print(f"Loaded {len(unstructured_csv_docs)} document(s)")


3️⃣ UnstructuredCSVLoader - Table Based Processing
Loaded 1 document(s)


In [13]:
print("\nContent:")
print(unstructured_csv_docs[0].page_content)
print("\nMetadata:")
print(unstructured_csv_docs[0].metadata)


Content:
Product Category Price Stock Description Laptop Electronics 999.99 50 High-performance laptop with 16GB RAM and 512GB SSD Mouse Accessories 29.99 200 Wireless optical mouse with ergonomic design Keyboard Accessories 79.99 150 Mechanical keyboard with RGB backlighting Monitor Electronics 299.99 75 27-inch 4K monitor with HDR support Webcam Electronics 89.99 100 1080p webcam with noise cancellation

Metadata:
{'source': 'E:/RAG/data/course_samples/structured_files/products.csv', 'file_directory': 'E:/RAG/data/course_samples/structured_files', 'filename': 'products.csv', 'last_modified': '2026-09-26T19:22:23', 'text_as_html': '<table><tr><td>Product</td><td>Category</td><td>Price</td><td>Stock</td><td>Description</td></tr><tr><td>Laptop</td><td>Electronics</td><td>999.99</td><td>50</td><td>High-performance laptop with 16GB RAM and 512GB SSD</td></tr><tr><td>Mouse</td><td>Accessories</td><td>29.99</td><td>200</td><td>Wireless optical mouse with ergonomic design</td></tr><tr><td>K

In [14]:
print("\nMetadata keys:")
print(unstructured_csv_docs[0].metadata.keys())


Metadata keys:
dict_keys(['source', 'file_directory', 'filename', 'last_modified', 'text_as_html', 'languages', 'filetype', 'category', 'element_id'])


In [15]:
if "text_as_html" in unstructured_csv_docs[0].metadata:
    print("\nHTML table representation:")
    print(unstructured_csv_docs[0].metadata["text_as_html"])
else:
    print("\nNo HTML table representation found.")


HTML table representation:
<table><tr><td>Product</td><td>Category</td><td>Price</td><td>Stock</td><td>Description</td></tr><tr><td>Laptop</td><td>Electronics</td><td>999.99</td><td>50</td><td>High-performance laptop with 16GB RAM and 512GB SSD</td></tr><tr><td>Mouse</td><td>Accessories</td><td>29.99</td><td>200</td><td>Wireless optical mouse with ergonomic design</td></tr><tr><td>Keyboard</td><td>Accessories</td><td>79.99</td><td>150</td><td>Mechanical keyboard with RGB backlighting</td></tr><tr><td>Monitor</td><td>Electronics</td><td>299.99</td><td>75</td><td>27-inch 4K monitor with HDR support</td></tr><tr><td>Webcam</td><td>Electronics</td><td>89.99</td><td>100</td><td>1080p webcam with noise cancellation</td></tr></table>


## Comparing CSVLoader & UnstructuredCSVLoader CSV Loaders

Now I’m comparing the output from both loaders.

I want to look at the difference in the number of documents and how the content is represented.

This is important because the loader changes the structure of the data before it reaches chunking and retrieval.

In [16]:
print("CSV LOADER COMPARISON")

print("\nCSVLoader:")
print(f"Documents created: {len(csv_docs)}")

print("\nUnstructuredCSVLoader:")
print(f"Documents created: {len(unstructured_csv_docs)}")

CSV LOADER COMPARISON

CSVLoader:
Documents created: 5

UnstructuredCSVLoader:
Documents created: 1


In [17]:
print("\nCSVLoader first document:")
print(csv_docs[0].page_content)

print("\n" + "=" * 60)

print("\nUnstructuredCSVLoader document:")
print(unstructured_csv_docs[0].page_content)


CSVLoader first document:
Product: Laptop
Category: Electronics
Price: 999.99
Stock: 50
Description: High-performance laptop with 16GB RAM and 512GB SSD


UnstructuredCSVLoader document:
Product Category Price Stock Description Laptop Electronics 999.99 50 High-performance laptop with 16GB RAM and 512GB SSD Mouse Accessories 29.99 200 Wireless optical mouse with ergonomic design Keyboard Accessories 79.99 150 Mechanical keyboard with RGB backlighting Monitor Electronics 299.99 75 27-inch 4K monitor with HDR support Webcam Electronics 89.99 100 1080p webcam with noise cancellation


### What I noticed

The two loaders represent the same CSV in different ways.

`CSVLoader` creates a separate document for each row.

`UnstructuredCSVLoader` treats the CSV more like a table and can keep a table-oriented representation.

This difference can matter depending on the question I want my RAG system to answer.

For row-level searches, a row-based representation can be useful.

For questions that depend on the table as a whole, preserving the table structure can be more useful.

## Why This Matters in RAG

Imagine I have a CSV containing product inventory.

A user may ask:

- What is the price of the Laptop?
- Which products are in the Electronics category?
- Show me the inventory table.
- Which products have the highest stock?
- Give me information from the complete product table.

For a row-level question, separate documents for each row can be useful.

For questions that depend on relationships across the table, keeping the table representation may be more useful.

This is why I should choose the ingestion format based on the type of questions my RAG system needs to answer.

In [18]:
print("Row-based documents from CSVLoader:", len(csv_docs))
print("Table-based document from UnstructuredCSVLoader:",
      len(unstructured_csv_docs))

print("\nCSVLoader metadata:")
print(csv_docs[0].metadata)

print("\nUnstructuredCSVLoader metadata:")
print(unstructured_csv_docs[0].metadata)

Row-based documents from CSVLoader: 5
Table-based document from UnstructuredCSVLoader: 1

CSVLoader metadata:
{'source': 'E:/RAG/data/course_samples/structured_files/products.csv', 'row': 0}

UnstructuredCSVLoader metadata:
{'source': 'E:/RAG/data/course_samples/structured_files/products.csv', 'file_directory': 'E:/RAG/data/course_samples/structured_files', 'filename': 'products.csv', 'last_modified': '2026-09-26T19:22:23', 'text_as_html': '<table><tr><td>Product</td><td>Category</td><td>Price</td><td>Stock</td><td>Description</td></tr><tr><td>Laptop</td><td>Electronics</td><td>999.99</td><td>50</td><td>High-performance laptop with 16GB RAM and 512GB SSD</td></tr><tr><td>Mouse</td><td>Accessories</td><td>29.99</td><td>200</td><td>Wireless optical mouse with ergonomic design</td></tr><tr><td>Keyboard</td><td>Accessories</td><td>79.99</td><td>150</td><td>Mechanical keyboard with RGB backlighting</td></tr><tr><td>Monitor</td><td>Electronics</td><td>299.99</td><td>75</td><td>27-inch 4K mon

## My Takeaway

I tested two different ways of ingesting the same CSV file.

**CSVLoader**
→ Converts rows into separate documents.

**UnstructuredCSVLoader**
→ Preserves the CSV more as a table representation.

The important lesson for me is that ingestion is not just about reading a file.

I also need to think about how I want the data to be represented before chunking, embedding, and retrieval.

The representation I choose should match the kind of questions my RAG system needs to answer.

## Excel Processing

Excel files are slightly different from CSV files because one workbook can contain multiple sheets.

So here I want to understand how to process each sheet while keeping information about the sheet itself.

I’ll first use Pandas because it gives me direct control over the workbook and its sheets.

## 1. Using Pandas for Excel Processing

I’m using Pandas to read the Excel workbook and process each sheet separately.

For every sheet, I want to create one LangChain `Document`.

I’ll also store useful information in metadata, such as:

- Sheet name
- Number of rows
- Number of columns
- Source file

In [19]:
# Method 1: Using pandas for full control
print("1️⃣ Pandas-based Excel Processing")


def process_excel_with_pandas(filepath: str) -> List[Document]:
    """Process Excel with sheet awareness"""

    documents = []

    excel_file = pd.ExcelFile(filepath)

    for sheet_name in excel_file.sheet_names:

        df = pd.read_excel(
            filepath,
            sheet_name=sheet_name
        )

        sheet_content = f"Sheet: {sheet_name}\n"
        sheet_content += f"Columns: {', '.join(df.columns)}\n"
        sheet_content += f"Rows: {len(df)}\n\n"
        sheet_content += df.to_string(index=False)

        doc = Document(
            page_content=sheet_content,
            metadata={
                'source': filepath,
                'sheet_name': sheet_name,
                'num_rows': len(df),
                'num_columns': len(df.columns),
                'data_type': 'excel_sheet'
            }
        )

        documents.append(doc)

    return documents

1️⃣ Pandas-based Excel Processing


### What I noticed

The important difference here is that I’m keeping the sheet name in the metadata.

This matters because an Excel workbook can contain many different sheets, and later I may need to know exactly which sheet a retrieved piece of information came from.

## Processing the Excel Workbook

Now I’m running the Pandas-based processor on the workbook.

Since the workbook has two sheets, I expect the processor to create one document for each sheet.

In [20]:
excel_docs = process_excel_with_pandas(
    'E:/RAG/data/course_samples/structured_files/inventory.xlsx'
)

print(f"Processed {len(excel_docs)} sheets")

Processed 2 sheets


## Looking at the Excel Documents

I’m printing the documents here so I can see how each Excel sheet was converted into a LangChain `Document`.

I’m mainly checking the content and metadata.

In [21]:
excel_docs

[Document(metadata={'source': 'E:/RAG/data/course_samples/structured_files/inventory.xlsx', 'sheet_name': 'Products', 'num_rows': 5, 'num_columns': 5, 'data_type': 'excel_sheet'}, page_content='Sheet: Products\nColumns: Product, Category, Price, Stock, Description\nRows: 5\n\n Product    Category  Price  Stock                                         Description\n  Laptop Electronics 999.99     50 High-performance laptop with 16GB RAM and 512GB SSD\n   Mouse Accessories  29.99    200        Wireless optical mouse with ergonomic design\nKeyboard Accessories  79.99    150           Mechanical keyboard with RGB backlighting\n Monitor Electronics 299.99     75                 27-inch 4K monitor with HDR support\n  Webcam Electronics  89.99    100                1080p webcam with noise cancellation'),
 Document(metadata={'source': 'E:/RAG/data/course_samples/structured_files/inventory.xlsx', 'sheet_name': 'Summary', 'num_rows': 2, 'num_columns': 3, 'data_type': 'excel_sheet'}, page_content='

### What I noticed

Each Excel sheet has become a separate document.

The document contains the sheet data as text, while the metadata keeps information about the original workbook and sheet.

This gives me a simple way to preserve the structure of the workbook before moving into chunking and retrieval.

## 2. Using UnstructuredExcelLoader

Now I’m trying another approach with `UnstructuredExcelLoader`.

The goal is to see how an unstructured loader handles the Excel workbook compared with my Pandas-based approach.

This can be useful when I want a loader to handle more complex document structures for me.

In [22]:
from langchain_community.document_loaders import UnstructuredExcelLoader

In [ ]:
# Method 2: UnstructuredExcelLoader
print("\n2️⃣UnstructuredExcelLoader")

try:
    excel_loader = UnstructuredExcelLoader(
        'E:/RAG/data/course_samples/structured_files/inventory.xlsx',
        mode="elements"
    )

    unstructured_docs = excel_loader.load()

    print("  ✅ Handles complex Excel features")
    print("  ✅ Preserves formatting info")
    print("  ❌ Requires unstructured library")
except Exception as e:
    print("  ℹ️ Requires unstructured library with Excel support")


2️⃣UnstructuredExcelLoader
  ✅ Handles complex Excel features
  ✅ Preserves formatting info
  ❌ Requires unstructured library


## Looking at the Unstructured Output

I’m checking the output here to understand what the loader actually created.

This is important because I don’t want to assume that two loaders will represent the same Excel file in exactly the same way.

In [24]:
unstructured_docs

[Document(metadata={'source': 'E:/RAG/data/course_samples/structured_files/inventory.xlsx', 'file_directory': 'E:/RAG/data/course_samples/structured_files', 'filename': 'inventory.xlsx', 'last_modified': '2026-09-26T19:22:24', 'page_name': 'Products', 'page_number': 1, 'text_as_html': '<table><tr><td>Product</td><td>Category</td><td>Price</td><td>Stock</td><td>Description</td></tr><tr><td>Laptop</td><td>Electronics</td><td>999.99</td><td>50</td><td>High-performance laptop with 16GB RAM and 512GB SSD</td></tr><tr><td>Mouse</td><td>Accessories</td><td>29.99</td><td>200</td><td>Wireless optical mouse with ergonomic design</td></tr><tr><td>Keyboard</td><td>Accessories</td><td>79.99</td><td>150</td><td>Mechanical keyboard with RGB backlighting</td></tr><tr><td>Monitor</td><td>Electronics</td><td>299.99</td><td>75</td><td>27-inch 4K monitor with HDR support</td></tr><tr><td>Webcam</td><td>Electronics</td><td>89.99</td><td>100</td><td>1080p webcam with noise cancellation</td></tr></table>', '

## My Takeaway

CSV and Excel files already have structure, so the way I convert them into documents is different from plain text and PDF files.

In this notebook, I learned:

- How `CSVLoader` converts rows into documents
- How to create my own CSV processing logic
- How to preserve useful metadata
- How to process multiple Excel sheets with Pandas
- How to use `UnstructuredExcelLoader`
- Why the structure of the original data matters during ingestion

The main flow I’m taking from this notebook is:

**CSV → Rows → Documents → Metadata**

**Excel → Sheets → Documents → Metadata**

The next important step is to see how these structured documents should be chunked and eventually embedded for retrieval.